# 13 - The 9 Essential Pipeline Monitoring Metrics

In production data engineering, building the pipeline is only half the job. The other half is **observability**: knowing the moment data drops, drifts, duplicates, or lags before downstream stakeholders notice.

### The 9 Essential Production Metrics Matrix:
| # | Metric | What You Check | Production Example |
| :--- | :--- | :--- | :--- |
| **1** | **Record Count** | Row volume across Medallion stages | Bronze has 12,056 rows; Silver has 11,981 |
| **2** | **Null Percentage** | Missing values in critical business columns | 0% null `claim_id`, 9.7% null `paid_amount` |
| **3** | **Duplicate Percentage** | Duplicated business primary keys | 2.87% duplicate `claim_id` in Bronze |
| **4** | **Freshness & Lag** | Wall-clock elapsed time since latest data arrival | Batch age is 69 hours (>24h SLA breach) |
| **5** | **Schema Drift** | Unexpected columns, dropped keys, or type mutations | `paid_amount` type changes from DOUBLE to VARCHAR |
| **6** | **Rejected Records** | Count and rate of rows routed to quarantine | 236 records quarantined (1.9% reject rate) |
| **7** | **Reconciliation** | Source vs target count & financial tie-backs | Claims paid ($36.9M) vs settled payments ($35.4M) |
| **8** | **Pipeline Success / Failure** | Job exit states, exception capture, and retries | Batch `2025-07-01` succeeded; task timeout alerted |
| **9** | **Processing Latency** | Elapsed execution time per stage vs SLA baseline | Ingestion took 18s (within 60s SLA threshold) |


In [2]:
import sys, pathlib, os, json, time, datetime
import pandas as pd
from pyspark.sql import functions as F, types as T

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)

spark = get_spark("13-monitoring-metrics")
dbutils = get_dbutils()
print("Setup complete. DuckDB warehouse and PySpark session ready.")


Setup complete. DuckDB warehouse and PySpark session ready.


---
### Metric 1: Record Count / Stage Volume Tracking

**What it monitors:** The row count at each boundary of the Medallion architecture (`Raw` $\to$ `Bronze` $\to$ `Silver` $\to$ `Gold`).
* **Why it matters:** A sudden 50% drop between Bronze and Silver indicates silent filtering or aggressive inner joins dropping rows.
* **Alert Rule:** Alert if row drop between adjacent stages exceeds threshold (e.g. `(Bronze - Silver) / Bronze > 5%`).


In [ ]:
# Metric 1: Tracking Row Volume Across Medallion Stages
record_counts = q("""
SELECT '1. raw.claims_api_records'  AS stage, COUNT(*) AS row_count FROM raw.claims_api_records
UNION ALL
SELECT '2. bronze.claims',          COUNT(*) FROM bronze.claims
UNION ALL
SELECT '3. silver.claims',          COUNT(*) FROM silver.claims
UNION ALL
SELECT '4. gold.monthly_claims_kpi',COUNT(*) FROM gold.monthly_claims_kpi
""")

# Calculate retention rate relative to Bronze
bronze_cnt = record_counts.loc[record_counts['stage'] == '2. bronze.claims', 'row_count'].values[0]
record_counts['pct_of_bronze'] = (record_counts['row_count'] / bronze_cnt * 100).round(2)

print("Metric 1 - Medallion Stage Record Counts:")
print(record_counts.to_string(index=False))


---
### Metric 2: Null Percentage / Data Completeness

**What it monitors:** The percentage of `NULL` values in required identifier columns and critical metric attributes.
* **Why it matters:** Primary keys must have **0%** nulls. If `claim_id` is null, records cannot be joined or aggregated.
* **Alert Rule:**
  * Primary keys: Alert if `null_pct > 0.0%` (Critical Incident).
  * Business metrics: Alert if `null_pct > baseline` (e.g. `billed_amount` null rate > 1.0%).


In [ ]:
# Metric 2: Calculating Null Percentages Across Columns
null_metrics = q("""
SELECT 
    COUNT(*) AS total_rows,
    -- Identifier column (Must be strictly 0%)
    ROUND(100.0 * COUNT(*) FILTER (WHERE claim_id IS NULL) / COUNT(*), 2) AS null_claim_id_pct,
    -- Foreign keys
    ROUND(100.0 * COUNT(*) FILTER (WHERE member_id IS NULL) / COUNT(*), 2) AS null_member_id_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE provider_id IS NULL) / COUNT(*), 2) AS null_provider_id_pct,
    -- Financial metric columns
    ROUND(100.0 * COUNT(*) FILTER (WHERE billed_amount IS NULL) / COUNT(*), 2) AS null_billed_amt_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE paid_amount IS NULL) / COUNT(*), 2) AS null_paid_amt_pct
FROM silver.claims
""")

print("Metric 2 - Silver Claims Null Rates (%):")
print(null_metrics.to_string(index=False))
print()
print("Interpretation: claim_id, member_id, and provider_id are 100% complete (0% null).")
print("paid_amount has 9.68% nulls because unpaid/pending claims have legitimate NULL adjudication.")


---
### Metric 3: Duplicate Percentage / Business Key Uniqueness

**What it monitors:** The proportion of duplicate business primary keys (`claim_id`).
* **Formula:** `100.0 * (COUNT(*) - COUNT(DISTINCT key)) / COUNT(*)`
* **Production Reality:**
  * In **Bronze:** Duplicates are expected when late updates or unmitigated batch replays occur.
  * In **Silver:** Duplicates must be strictly **0.00%**.


In [ ]:
# Metric 3: Duplicate Rate Comparison (Bronze vs Silver)
dup_report = q("""
SELECT 
    'bronze.claims' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT claim_id) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT claim_id) AS duplicate_rows,
    ROUND(100.0 * (COUNT(*) - COUNT(DISTINCT claim_id)) / COUNT(*), 2) AS duplicate_pct,
    CASE WHEN COUNT(*) = COUNT(DISTINCT claim_id) THEN 'PASS' ELSE 'WARN (Replays Present)' END AS status
FROM bronze.claims
UNION ALL
SELECT 
    'silver.claims',
    COUNT(*),
    COUNT(DISTINCT claim_id),
    COUNT(*) - COUNT(DISTINCT claim_id),
    ROUND(100.0 * (COUNT(*) - COUNT(DISTINCT claim_id)) / COUNT(*), 2),
    CASE WHEN COUNT(*) = COUNT(DISTINCT claim_id) THEN 'PASS' ELSE 'FAIL (Key Violated)' END
FROM silver.claims
""")

print("Metric 3 - Duplicate Key Health:")
print(dup_report.to_string(index=False))


---
### Metric 4: Freshness & Data Lag (SLA Monitoring)

**What it monitors:** The time elapsed between the latest record timestamp (`_loaded_at` or `max(event_time)`) and the current wall-clock time (`now()`).
* **Why it matters:** If daily batches stop arriving, row count checks on yesterday's data will pass while downstream reports display stale data.
* **Alert Rule:** If `date_diff('hour', max_loaded_at, now()) > SLA_HOURS`, trigger a P1 Freshness Alert.


In [ ]:


freshness_check = q("""
SELECT 
    MAX(_loaded_at) AS latest_load_timestamp,
    now() AS current_system_time,
    date_diff('hour', MAX(_loaded_at), now()) AS lag_hours,
    date_diff('day', MAX(_loaded_at), now()) AS lag_days,
    24 AS sla_threshold_hours,
    CASE 
        WHEN date_diff('hour', MAX(_loaded_at), now()) <= 24 THEN 'HEALTHY (Within SLA)'
        ELSE 'ALERT: Freshness SLA Breached (>24h lag)'
    END AS sla_status
FROM bronze.claims
""")

print("Metric 4 - Data Freshness & Pipeline Lag:")
print(freshness_check.to_string(index=False))


---
### Metric 5: Schema Drift & Data Type Integrity

**What it monitors:** Upstream contract mutations—such as added columns, renamed fields, or data type casting changes (e.g. `paid_amount` shifting from `DOUBLE` to `VARCHAR`).
* **Implementation:** Compare current table schema against the registered JSON schema contract.


In [3]:
# Metric 5: Schema Contract Assertion & Drift Detection
CONTRACT_SCHEMA = {
    "claim_id": "VARCHAR",
    "member_id": "VARCHAR",
    "provider_id": "VARCHAR",
    "claim_date": "DATE",
    "status": "VARCHAR",
    "billed_amount": "DECIMAL(12,2)",
    "paid_amount": "DECIMAL(12,2)"
}

def validate_schema(actual_schema: dict, contract_schema: dict):
    diffs = []
    for col_name, expected_type in contract_schema.items():
        if col_name not in actual_schema:
            diffs.append({"column": col_name, "issue": "MISSING_COLUMN", "expected": expected_type, "actual": "None"})
        elif actual_schema[col_name] != expected_type:
            diffs.append({"column": col_name, "issue": "TYPE_MUTATION", "expected": expected_type, "actual": actual_schema[col_name]})
    return diffs

# 1. Validate actual Silver table against contract
actual_schema_df = q("""
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_schema = 'silver' AND table_name = 'claims'
""")
actual_schema = dict(zip(actual_schema_df['column_name'], actual_schema_df['data_type']))
diffs = validate_schema(actual_schema, CONTRACT_SCHEMA)

print(f"Metric 5 - Production Schema Check: {'PASS (0 contract violations)' if not diffs else 'FAIL'}")

# 2. Simulate Upstream Drift Detection
simulated_drift = dict(actual_schema)
simulated_drift["paid_amount"] = "VARCHAR"  # Upstream source accidentally sent string currency
drift_report = validate_schema(simulated_drift, CONTRACT_SCHEMA)

print()
print("Simulated Drift Alert (When upstream sends VARCHAR instead of DECIMAL):")
print(pd.DataFrame(drift_report).to_string(index=False))


Metric 5 - Production Schema Check: PASS (0 contract violations)

Simulated Drift Alert (When upstream sends VARCHAR instead of DECIMAL):
     column         issue      expected  actual
paid_amount TYPE_MUTATION DECIMAL(12,2) VARCHAR


---
### Metric 6: Rejected Records & Quarantine Rate

**What it monitors:** The number and percentage of records diverted to `dq.quarantine` due to contract violations.
* **Why it matters:** A sudden spike in rejected records indicates upstream source changes or corrupted upstream batches.
* **Formula:** `Rejection Rate = (Quarantined Records / Total Ingested Records) * 100`


In [ ]:
# Metric 6: Quarantine Rate & Defect Breakdown
quarantine_summary = q("""
SELECT 
    (SELECT COUNT(*) FROM dq.quarantine) AS quarantined_rows,
    (SELECT COUNT(*) FROM raw.claims_api_records) AS total_raw_ingested,
    ROUND(100.0 * (SELECT COUNT(*) FROM dq.quarantine) / (SELECT COUNT(*) FROM raw.claims_api_records), 2) AS reject_rate_pct,
    5.0 AS max_allowed_reject_rate_pct,
    CASE 
        WHEN 100.0 * (SELECT COUNT(*) FROM dq.quarantine) / (SELECT COUNT(*) FROM raw.claims_api_records) <= 5.0 
        THEN 'PASS (Normal Quarantine Rate)'
        ELSE 'ALERT: Elevated Quarantine Volume'
    END AS status
""")

print("Metric 6 - Quarantine Health Summary:")
print(quarantine_summary.to_string(index=False))

print()
print("Top Defect Reasons in Quarantine:")
print(q("""
SELECT source, reason, COUNT(*) AS defect_count,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dq.quarantine), 1) AS pct_of_quarantine
FROM dq.quarantine 
GROUP BY 1, 2 
ORDER BY defect_count DESC
""").to_string(index=False))


---
### Metric 7: Cross-Stage & Financial Reconciliation

**What it monitors:** Consistency between source events and target financial ledgers.
* **The Rule:** Reconcile dollars, not just rows. Compare like-for-like (`claims.paid_amount` against **settled** payments, excluding voided entries).
* **Formula:** `Financial Variance = SUM(Claims Paid) - SUM(Settled Payments)`


In [ ]:
# Metric 7: Financial Tie-Back Reconciliation
fin_recon = q("""
SELECT 
    (SELECT ROUND(SUM(paid_amount), 2) FROM silver.claims) AS source_claims_paid,
    (SELECT ROUND(SUM(payment_amount), 2) FROM silver.payments WHERE payment_status = 'Paid') AS target_settled_payments,
    (SELECT ROUND(SUM(payment_amount), 2) FROM silver.payments WHERE payment_status = 'Voided') AS voided_payments_excluded,
    ROUND((SELECT SUM(paid_amount) FROM silver.claims) - 
          (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Paid'), 2) AS dollar_variance,
    ROUND(100.0 * ((SELECT SUM(paid_amount) FROM silver.claims) - 
                   (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Paid')) / 
          (SELECT SUM(paid_amount) FROM silver.claims), 2) AS variance_pct
""")

print("Metric 7 - Cross-System Financial Reconciliation:")
print(fin_recon.to_string(index=False))
print()
print("Audit Note: $1.49M variance (4.04%) is caused by 486 approved claims in timing transit not yet disbursed.")


---
### Metric 8: Pipeline Success / Failure State Tracking

**What it monitors:** Job execution state, error codes, and retry counts across orchestration tasks (e.g., Airflow, Databricks Workflows).
* **Best Practice:** Maintain an immutable pipeline execution audit log table logging `batch_id`, `start_time`, `end_time`, `status`, and `records_written`.


In [ ]:
# Metric 8: Simulated Pipeline Audit Ledger
# In production, this ledger is written by your orchestrator or Medallion write job
pipeline_audit_ledger = [
    {"batch_id": "2025-07-01T02:05:00Z", "task_name": "ingest_claims", "status": "SUCCESS", "records": 12056, "retries": 0},
    {"batch_id": "2025-07-01T02:05:00Z", "task_name": "conform_silver", "status": "SUCCESS", "records": 11981, "retries": 0},
    {"batch_id": "2025-07-01T02:05:00Z", "task_name": "build_gold_kpi", "status": "SUCCESS", "records": 18,    "retries": 0},
    {"batch_id": "2025-04-16T02:05:00Z", "task_name": "ingest_claims", "status": "SUCCESS", "records": 302,   "retries": 1},
    {"batch_id": "2025-04-16T02:05:00Z", "task_name": "conform_silver", "status": "FAILED",  "records": 0,     "retries": 3},
]

df_audit = pd.DataFrame(pipeline_audit_ledger)
failed_tasks = df_audit[df_audit['status'] == 'FAILED']

print("Metric 8 - Pipeline Task Execution Ledger:")
print(df_audit.to_string(index=False))
print()
if not failed_tasks.empty:
    print(f"ALERT: {len(failed_tasks)} failed task(s) detected in execution ledger:")
    for _, row in failed_tasks.iterrows():
        print(f"  -> Batch {row['batch_id']}: Task '{row['task_name']}' failed after {row['retries']} retries.")
else:
    print("All tasks succeeded cleanly.")


---
### Metric 9: Processing Latency & Runtime Duration

**What it monitors:** The wall-clock execution time taken by each pipeline stage.
* **Why it matters:** If a job typically finishes in 20 minutes but takes 90 minutes today, you have data skew, cluster throttling, or unpruned partitions causing spill.
* **Alert Rule:** If `runtime_seconds > 2 * baseline_average_seconds`, trigger an SLA degradation warning.


In [ ]:
# Metric 9: Benchmarking Job Execution Duration
def time_query_execution(sql_query: str):
    start = time.perf_counter()
    res = q(sql_query)
    elapsed = time.perf_counter() - start
    return len(res), elapsed

rows, latency_sec = time_query_execution("""
SELECT 
    date_trunc('month', claim_date) AS month, 
    COUNT(*) AS claims, 
    ROUND(SUM(paid_amount), 2) AS total_paid
FROM silver.claims 
GROUP BY 1 
ORDER BY 1
""")

SLA_MAX_LATENCY_SEC = 5.0  # Simulated SLA limit
print(f"Metric 9 - Stage Processing Latency: {latency_sec:.3f} seconds ({rows} rows produced)")
print(f"SLA Baseline Target:                < {SLA_MAX_LATENCY_SEC:.1f} seconds")
print(f"Status:                             {'PASS (Within SLA)' if latency_sec <= SLA_MAX_LATENCY_SEC else 'WARN (Degraded Performance)'}")


---
### The Unified 9-Metric Enterprise Health Scorecard

In production, these 9 metrics are collected at the end of every pipeline run and published to your observability dashboard (e.g. Databricks Lakehouse Monitoring, Datadog, Grafana, or Great Expectations).


In [ ]:
# Assembling the Complete Pipeline Health Scorecard
dashboard_summary = [
    {"metric_id": 1, "metric_name": "Record Count",        "value": f"{bronze_cnt:,} (Bronze) -> {len(q('SELECT * FROM silver.claims')):,} (Silver)", "status": "PASS"},
    {"metric_id": 2, "metric_name": "Null Percentage",     "value": f"0.0% PK nulls | 9.68% paid_amount nulls", "status": "PASS"},
    {"metric_id": 3, "metric_name": "Duplicate Rate",      "value": f"{dup_report.loc[dup_report['table_name']=='silver.claims', 'duplicate_pct'].values[0]}% in Silver", "status": "PASS"},
    {"metric_id": 4, "metric_name": "Freshness & Lag",     "value": f"{freshness_check['lag_hours'].values[0]} hours lag", "status": "WARN (Historical DB)"},
    {"metric_id": 5, "metric_name": "Schema Integrity",    "value": "7/7 Contract Columns Valid", "status": "PASS"},
    {"metric_id": 6, "metric_name": "Quarantine Rate",     "value": f"{quarantine_summary['reject_rate_pct'].values[0]}% ({quarantine_summary['quarantined_rows'].values[0]} rows)", "status": "PASS"},
    {"metric_id": 7, "metric_name": "Reconciliation",      "value": f"${fin_recon['dollar_variance'].values[0]:,.2f} variance ({fin_recon['variance_pct'].values[0]}%)", "status": "WARN (Timing Lead)"},
    {"metric_id": 8, "metric_name": "Job Success State",   "value": "4/5 Tasks Succeeded (1 Failed Replay)", "status": "WARN"},
    {"metric_id": 9, "metric_name": "Processing Latency",  "value": f"{latency_sec:.3f}s (SLA < 5.0s)", "status": "PASS"},
]

df_scorecard = pd.DataFrame(dashboard_summary)
print("=========================================================================================")
print("                   ENTERPRISE PIPELINE OBSERVABILITY SCORECARD                           ")
print("=========================================================================================")
print(df_scorecard.to_string(index=False))
print("=========================================================================================")
